# 16.15 同一份材料問很多題，怎樣少算又不互相污染？


同一份客服材料先問「現在要辦什麼？」（Q1），再問「哪件事已完成？」（Q2），最後重問Q1。如果材料沒變，第三次不應因為剛問過Q2而改答案。我們想省下每題重讀材料的工作，同時保住每題獨立的計算結果。

前置是[9.17的候選計分](https://birdhackor.github.io/tiny-perceptron-vlm/9.17.html)、[16.2–16.3的KV快取與重算核對](https://birdhackor.github.io/tiny-perceptron-vlm/16.2.html)、[16.5的讀取隔離](https://birdhackor.github.io/tiny-perceptron-vlm/16.5.html)。把本課有限的機率判斷器記為C。9.12的獨立候選評分仍可每次重讀材料；現在改用**只讀材料的編碼器（state-only encoder）**產生記憶矩陣`M = E(state)`。它完全不讀問題或候選；每個問題、實質候選用自己的query讀M，算出各自的特徵和分數。這種query來自分支、Key與Value來自另一份材料的注意力，叫**交叉注意力（cross-attention）**。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/omni-16.15-readonly-memory.svg")))

圖中的唯讀是實際約定：分支可改自己的中間特徵，不能把結果寫回M，也不能替下一題保存自己的候選狀態。重算每題時本來都會得到同一個`E(state)`，所以可重用這部分。共享只免去材料編碼；新問題的query、注意力比對與候選評分仍要計算，是否真的省時間要另外量測。

這是**新架構**。舊版C把材料、問題、某候選接成一列，用雙向注意力編碼；材料位置也可能讀到後面的問題。因此不能從舊模型截出一塊「材料特徵」，直接叫它共享記憶體，還聲稱結果不變。需要改分工、重新訓練及驗收。

以下人工向量只示意依賴：`state`兩列是假設的材料特徵，兩軸對應本例卡片與地址線索；沒有文字編碼或學習。`encode`讓材料內部互相讀取，`answer`則讓每個候選的query交叉讀M。為了看清運算，投影都取恆等矩陣；最後none沿用9.17的人工對稱規則。


In [ ]:
import copy
import math
import torch

torch.set_num_threads(1)
state = torch.tensor([[1.0, 0.0], [0.0, 2.0]], dtype=torch.float32)
candidates = torch.eye(2)  # 依序card、address；none另由集合摘要產生
q1, q2 = torch.tensor([2.0, 0.0]), torch.tensor([0.0, 2.0])
questions = [q1, q2, q1]
calls = 0


def encode(x):
    global calls
    calls += 1
    weights = (x @ x.T / math.sqrt(2)).softmax(-1)
    return x + weights @ x


def answer(memory, question):
    query = question[None, :] + candidates
    # 本例K=V=M；兩列query各屬於一個實質候選。
    weights = (query @ memory.T / math.sqrt(2)).softmax(-1)
    hidden = candidates + weights @ memory
    real = (hidden * question).sum(-1) / math.sqrt(2)
    scores = torch.cat([real, (1.0 - real.max())[None]])
    return weights, hidden, scores, scores.softmax(-1)


fresh = []
for q in questions:
    m = encode(state)
    fresh.append((m, *answer(m, q)))
fresh_calls = calls
calls = 0
memory = encode(state)
saved = memory.clone()
shared = [(memory, *answer(memory, q)) for q in questions]
error = max(float((a - b).abs().max()) for f, s in zip(fresh, shared) for a, b in zip(f, s))
print("材料編碼次數：重算／共享", fresh_calls, calls)
print("M、權重、特徵、分數、機率的最大差", error)
print("Q1/Q2/Q1機率", [[round(v, 4) for v in s[-1].tolist()] for s in shared])
assert error <= 1e-5 and torch.equal(memory, saved)


# 負控制：故意讓每個分支把自己的query加回M的第一列。
def dirty_answer(m, q):
    result = answer(m, q)
    m[0].add_(q)  # 錯誤的寫回；正確answer沒有這一行
    return result


dirty = memory.clone()
first = dirty_answer(dirty, q1)
dirty_answer(dirty, q2)
last = dirty_answer(dirty, q1)
print("錯誤寫回後Q1機率差", round(float((first[-1] - last[-1]).abs().max()), 4))

# 雙向混合的反例：問題也進encode時，材料列本身就依賴問題。
old1 = encode(torch.cat([state, q1[None, :]]))[:2]
old2 = encode(torch.cat([state, q2[None, :]]))[:2]
print("雙向混合示意的材料特徵差", round(float((old1 - old2).abs().max()), 4))


# 再用最小循環例子追蹤可變prefix；不是DeltaNet的完整公式。
def advance(cache, values):
    for x in values:
        cache["kv"].append((x, x))  # 人工K=x、V=x，保留逐位置紀錄
        filtered = x + 0.1 * cache["conv"][-1]
        cache["conv"][:] = [x]  # 短卷積保留前一個輸入
        cache["r"] = 0.5 * cache["r"] + filtered
    return cache["r"]


prefix = {"kv": [], "conv": [0.0], "r": 0.0}
advance(prefix, [1.0, 2.0])  # 先讀材料，得到共同起點
clean = [advance(copy.deepcopy(prefix), [x]) for x in [3.0, 4.0, 3.0]]
dirty_prefix = copy.deepcopy(prefix)
polluted = [advance(dirty_prefix, [x]) for x in [3.0, 4.0, 3.0]]
print("各分支從prefix起步", [round(v, 4) for v in clean])
print("誤用前題結尾接下一題", [round(v, 4) for v in polluted])

關鍵輸出如下；每組機率依序是card、address、none：

```text
材料編碼次數：重算／共享 3 1
M、權重、特徵、分數、機率的最大差 0.0
Q1/Q2/Q1機率 [[0.9216, 0.0769, 0.0015], [0.1746, 0.8254, 0.0], [0.9216, 0.0769, 0.0015]]
錯誤寫回後Q1機率差 0.1079
雙向混合示意的材料特徵差 0.9325
各分支從prefix起步 [4.5, 5.5, 4.5]
誤用前題結尾接下一題 [4.5, 6.55, 6.675]
```

共享版前後Q1相同，Q2的none印成0.0只是取四位小數。刻意寫回M後，最後Q1與首次Q1的機率差0.1079；這個**負控制**故意犯錯，確認檢查能抓到我們擔心的污染。雙向混合的材料特徵差0.9325，指出為什麼不能照搬舊計算的局部結果。

這段循環例子用`r ← 0.5r + filtered`壓縮已讀內容，`conv`保留短濾波還需要的最近輸入；`kv`則保留各位置的Key與Value，本段只列存放，不再做注意力。它們是三種不同的記錄：注意力拿新query回頭比對KV；循環層拿舊r計算新r；短卷積也需要自己的歷史輸入。刪掉KV尾端不會把已更新的r倒回去。

Kev-4B的固定底座配置有24層線性循環注意力與8層完整注意力；前者採Gated DeltaNet，這裡只需知道它會更新循環狀態及卷積狀態，並非本例的簡單加法。普通注意力遮罩只管讀哪些Key，不能阻止這些狀態把前題帶到後題。其公開服務讓每題各跑一列，並保護共同prefix：每個分支都從相同的KV、卷積與循環狀態起步。這和新C唯讀M是兩條不同的共享路徑，不能混稱「只快取KV就夠」。

正式快取只在凍結推論時使用。快取身分要綁定材料的實際bytes、編碼器權重、tokenizer與前處理版本、padding及位置規則、dtype與device；任何一項改變就重建。固定FP32、評估模式且關閉dropout（[5.17](https://birdhackor.github.io/tiny-perceptron-vlm/5.17.html)說明評估模式如何關閉訓練時的隨機省略），再核對重算／共享的M、中間特徵與機率在`1e-5`內，及Q1→Q2→Q1不污染。這些等價檢查不證明候選判斷本身正確。

這裡的tokenizer是把材料轉成token／ID的規則；dtype是數值格式，例如本例的`torch.float32`；device是計算所在的裝置，例如本例的CPU或另一環境的GPU。記錄這些身分，才能分辨「文字相同」與「仍用同一套計算」：例如換了文字編號規則、改用另一數值格式或搬到另一裝置，都按上述規則重建快取。

練習分兩步，各自從原程式重新開始：先刪掉`dirty_answer`的`m[0].add_(q)`，預測Q1機率差歸零後執行；再只把`clean = ...`那行的`copy.deepcopy(prefix)`改成`prefix`，觀察共同起點如何被修改。說明兩處錯誤為什麼都不能靠「材料文字沒改」自動避免。

來源：[Kev固定版`_rows_hidden`與prefix分支](https://github.com/jaredpalmer/kev/blob/2074cc44256200bb37540103fa4572901faa10f3/kev/model.py#L340-L461)、[Kev-4B固定卡片的層型說明](https://huggingface.co/jaredpalmer/kev-4b/blob/6cfce5c2fa4b4bd64026336ab649c5ca78857d52/README.md#model-details)、[所釘底座的`layer_types`](https://huggingface.co/Qwen/Qwen3.5-4B-Base/blob/1001bb4d826a52d1f399e183466143f4da7b741b/config.json)、[舊C雙向`CandidateScorer`與`Block`](https://github.com/birdhackor/tiny-perceptron-vlm/blob/2b3332ca4392a946a065782815f9f60dae9a87c7/tiny_perceptron/lesson_products/model.py#L98-L177)。本節原創小例沒有重現DeltaNet或Kev權重推論。
